In [1]:
from pathlib import Path
import random
import shutil
import pandas as pd
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

random.seed(42)

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path("../data")

RAW_DIR = DATA_DIR / "raw"

FAIRFACE_DIR = RAW_DIR / "fairface"
CELEBA_DIR = RAW_DIR / "celeba"
DROWSINESS_DIR = RAW_DIR / "driver_drowsiness"
DISTRACTION_DIR = RAW_DIR / "driver_distraction"

for folder in [

    FAIRFACE_DIR,
    CELEBA_DIR,
    DROWSINESS_DIR,
    DISTRACTION_DIR

]:

    (folder/"images").mkdir(
        parents=True,
        exist_ok=True
    )

In [5]:
from pathlib import Path

PROJECT_DIR = Path("/workspaces/DBA-ai-trust-score-framework-Interim")

SAMPLE_DIR = PROJECT_DIR / "data" / "samples"

FAIRFACE_DIR = SAMPLE_DIR / "fairface"
CELEBA_DIR = SAMPLE_DIR / "celeba"
DROWSINESS_DIR = SAMPLE_DIR / "driver_drowsiness"
DISTRACTION_DIR = SAMPLE_DIR / "driver_distraction"

for folder in [
    FAIRFACE_DIR,
    CELEBA_DIR,
    DROWSINESS_DIR,
    DISTRACTION_DIR
]:

    (folder/"images").mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Folder structure created.")

✅ Folder structure created.


In [18]:
def save_dataset_sample(dataset, save_dir, dataset_name, image_column="image"):

    metadata = []

    image_dir = save_dir / "images"

    for idx, sample in enumerate(tqdm(dataset)):

        # Handle datasets with nested image field
        if image_column == "Image_data":
            image = sample["Image_data"]["file"]
        else:
            image = sample[image_column]

        filename = f"{dataset_name}_{idx:05d}.jpg"

        image.save(image_dir / filename)

        row = {}

        for key, value in sample.items():

            if key != image_column:

                row[key] = value

        row["file_name"] = filename

        metadata.append(row)

    metadata = pd.DataFrame(metadata)

    metadata.to_csv(
        save_dir / "metadata.csv",
        index=False
    )

    return metadata

In [7]:
import os
from huggingface_hub import login

token = os.getenv("HF_TOKEN")

if token:
    login(token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


_______________FAIRFACE DATA LOADING__________________________

In [8]:
from datasets import load_dataset

fairface = load_dataset(
    "HuggingFaceM4/FairFace",
    "0.25",          # configuration
    split="train"
)

fairface_sample = fairface.shuffle(seed=42).select(range(1000))

fairface_metadata = save_dataset_sample(

    fairface_sample,

    FAIRFACE_DIR,

    "fairface"

)

100%|██████████| 1000/1000 [00:01<00:00, 965.52it/s]

Saved 1000 images.


_____________________________CelebA DATA LOADING_________________________________

In [10]:
celeba = load_dataset(
    "student/celebA",
    split="train"
)

celeba_sample = celeba.shuffle(seed=42).select(range(400))

celeba_metadata = save_dataset_sample(

    celeba_sample,

    CELEBA_DIR,

    "celeba"

)

100%|██████████| 400/400 [16:13<00:00,  2.43s/it]

Saved 400 images.


_______________________________DRIVER DROWSINESS DATA LOADING____________________________

In [19]:
driver_drowsiness = load_dataset(
    "MichalMlodawski/closed-open-eyes",
    split ="train"

)

driver_drowsiness_sample = driver_drowsiness.shuffle(seed=42).select(range(1000))

driver_drowsiness_metadata = save_dataset_sample(
    driver_drowsiness_sample,
    DROWSINESS_DIR,
    "driver_drowsiness",
    image_column="Image_data"
)

100%|██████████| 1000/1000 [00:08<00:00, 117.59it/s]


______________________________________Driver Distraction________________________________

In [20]:
driver_distraction = load_dataset(
    "gymprathap/Driver-Distracted-Dataset",
    split="train"
)

driver_distraction_sample = driver_distraction.shuffle(seed=42).select(range(1000))

driver_distraction_metadata = save_dataset_sample(

    driver_distraction_sample,

    DISTRACTION_DIR,

    "driver_distraction"

)

100%|██████████| 1000/1000 [23:45<00:00,  1.43s/it]


In [21]:
print("FairFace :", len(fairface_metadata))
print("CelebA :", len(celeba_metadata))
print("Drowsiness :", len(driver_drowsiness_metadata))
print("Distraction :", len(driver_distraction_metadata))

FairFace : 1000
CelebA : 400
Drowsiness : 1000
Distraction : 1000
